# 🧪 InfoDev - Playground de Testes e Debug

Bem-vindo ao ambiente de desenvolvimento do **InfoDev**, um sistema multi-agente RAG projetado para extrair e raciocinar sobre dados de engenharia de software (issues, commits e e-mails).

Este notebook tem como objetivo testar os componentes do sistema de forma isolada (Módulos, Agentes e Bancos Vetoriais) antes de integrá-los no grafo de execução final (LangGraph).

### ⚙️ Configurações e Ambiente
* **Python Mínimo:** 3.11 (Garante compatibilidade com wheels C++ de IA)
* **Gerenciamento de Pacotes:** Certifique-se de que o seu ambiente virtual está ativo e instale as dependências com: `pip install -r requirements.txt`.
* **Variáveis de Ambiente:** O sistema depende do arquivo `.env` na raiz do projeto contendo as chaves `GROQ_API_KEY` e `MONGO_URI`.

In [1]:
# MÁGICA DO JUPYTER: Força o notebook a recarregar arquivos .py 
# automaticamente toda vez que você rodar uma célula.
%load_ext autoreload
%autoreload 2

import os
from dotenv import load_dotenv

# Garante que as credenciais do .env estão ativas no ambiente
load_dotenv(override=True)
print("Variáveis de ambiente carregadas.")

Failed to read module file 'C:\Users\mathe\AppData\Local\Programs\Python\Python311\Lib\shlex.py' for module 'shlex': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\projects\infodev\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\projects\infodev\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\mathe\AppData\Local\Programs\Python\Python311\Lib\importlib\__init__.py", line 126, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1204, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1176, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1140, in _find_and_load_un

Variáveis de ambiente carregadas.


## 📊 Area de Testes

Conexão com o banco e contagem de documentos

In [2]:
from pymongo import MongoClient

print("📊 --- Raio-X do Banco de Dados MongoDB ---")

# Conecta ao banco de dados (ajuste a URI se necessário)
MONGO_URI = "mongodb://localhost:27017/"
DB_NAME = "clean_shark"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

colecoes = ["rich_issues", "rich_commits", "rich_emails"]
projeto_alvo = "tez" # O projeto que você vai vetorizar

for col_name in colecoes:
    colecao = db[col_name]
    
    # Conta o total absoluto na coleção
    total_docs = colecao.count_documents({})
    
    # Conta apenas os documentos do projeto alvo
    total_projeto = colecao.count_documents({"project": projeto_alvo})
    
    # Busca um documento de exemplo para checar os campos
    exemplo = colecao.find_one()
    campos = list(exemplo.keys()) if exemplo else "Nenhum documento"
    
    # Tenta descobrir quais projetos únicos existem nessa coleção (pode demorar um pouquinho se o banco for gigante)
    try:
        projetos_unicos = colecao.distinct("project")
    except:
        projetos_unicos = ["Erro ao buscar"]
        
    print(f"\n📁 Coleção: {col_name}")
    print(f"  - Total de documentos: {total_docs}")
    print(f"  - Documentos do projeto '{projeto_alvo}': {total_projeto}")
    print(f"  - Projetos únicos encontrados: {projetos_unicos}")
    print(f"  - Campos disponíveis: {campos}")

print("\n✅ Diagnóstico concluído!")

📊 --- Raio-X do Banco de Dados MongoDB ---

📁 Coleção: rich_issues
  - Total de documentos: 22342
  - Documentos do projeto 'tez': 4133
  - Projetos únicos encontrados: ['nifi', 'pdfbox', 'phoenix', 'ranger', 'tez']
  - Campos disponíveis: ['_id', 'project', 'type', 'original_id', 'title', 'status', 'text_for_embedding', 'created_at']

📁 Coleção: rich_commits
  - Total de documentos: 33780
  - Documentos do projeto 'tez': 3659
  - Projetos únicos encontrados: ['nifi', 'pdfbox', 'phoenix', 'ranger', 'tez']
  - Campos disponíveis: ['_id', 'project', 'type', 'hash', 'date', 'text_for_embedding', 'files_touched']

📁 Coleção: rich_emails
  - Total de documentos: 225463
  - Documentos do projeto 'tez': 8942
  - Projetos únicos encontrados: ['nifi', 'pdfbox', 'phoenix', 'ranger', 'tez']
  - Campos disponíveis: ['_id', 'project', 'type', 'subject', 'text_for_embedding', 'date']

✅ Diagnóstico concluído!


## 📁 Inicialização dos Managers

In [3]:
import sys
sys.path.append('./src')

from Config import Config
from VectorStoreManager import VectorStoreManager

print("--- Inicializando as 3 Bases de Dados Vetoriais ---")

projeto_alvo = "tez"
filtro = {"project": projeto_alvo}
db_path = f"./vectorstores/{projeto_alvo}_db"

# Modelos escolhidos
MODELO_CODIGO = "jinaai/jina-embeddings-v2-base-code"
MODELO_TEXTO = "nomic-ai/nomic-embed-text-v1.5"

# 1. Instancia passando o modelo correto para cada domínio
manager_commits = VectorStoreManager(
    persist_directory=db_path, 
    collection_name="commits", 
    model_name=MODELO_CODIGO     # <--- Jina para código
)

manager_issues = VectorStoreManager(
    persist_directory=db_path, 
    collection_name="issues", 
    model_name=MODELO_TEXTO      # <--- Nomic para texto
)

manager_emails = VectorStoreManager(
    persist_directory=db_path, 
    collection_name="emails", 
    model_name=MODELO_TEXTO      # <--- Nomic para texto
)

print("\n✅ Bases de dados vetoriais inicializadas com sucesso!")

--- Inicializando as 3 Bases de Dados Vetoriais ---


c:\projects\infodev\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\projects\infodev\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\mathe\.cache\huggingface\modules\transformers_modules\jinaai\jina-bert-v2-qk-post-norm\3baf9e3ac750e76e8edd3019170176884695fb94\configuration_bert.py:29: UserWarning: optimum is not installed. To use OnnxConfig and BertOnnxConfig, make sure that `optimum` package is installed
  warnings.warn("optimum is not installed. To use OnnxConfig and BertOnnxConfig, make sure that `optimum` package is installed")
c:\projects\infodev\./src\VectorStoreManager.py:3


✅ Bases de dados vetoriais inicializadas com sucesso!


## 🍴 Ingestão
### ⚠️ Atenção ⚠️
A lógica de ingestão leva um tempo considerável. Caso tenha uma GPU, altere a variável "EMBEDDINGS_DEVICE" em src/Config.py

In [4]:
DO_INGESTION = True  # <--- Mude para True para rodar a ingestão (lembre-se de ajustar o filtro se necessário)

if DO_INGESTION:
    print("\n🚀 --- INICIANDO PIPELINE DE INGESTÃO ---")
    
    manager_commits.ingest_documents(
        mongo_collection_name=Config.COLLECTION_COMMITS, 
        doc_type="commit", 
        mongo_filter=filtro,
        batch_size=32
    )
    
    manager_issues.ingest_documents(
        mongo_collection_name=Config.COLLECTION_ISSUES, 
        doc_type="issue", 
        mongo_filter=filtro,
        batch_size=32
    )
    
    manager_emails.ingest_documents(
        mongo_collection_name=Config.COLLECTION_EMAILS, 
        doc_type="email", 
        mongo_filter=filtro,
        batch_size=32
    )
    print("\n✅ PIPELINE TOTAL CONCLUÍDO!")

Failed to read module file 'c:\projects\infodev\./src\Config.py' for module 'Config': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\projects\infodev\.venv\Lib\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 219, in update_sources
    self.source_by_modname[new_modname] = f.read()
                                          ^^^^^^^^
  File "C:\Users\mathe\AppData\Local\Programs\Python\Python311\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x9d in position 357: character maps to <undefined>



🚀 --- INICIANDO PIPELINE DE INGESTÃO ---

Conectando ao MongoDB (clean_shark -> rich_commits)...
Buscando documentos...
Dividindo 1000 documentos...
Total gerado: 15500 chunks.
Iniciando vetorização (Lotes de 32)...


Vetorizando 'commits': 100%|██████████| 485/485 [3:12:12<00:00, 23.78s/it]  


Ingestão na coleção 'commits' concluída com sucesso!

Conectando ao MongoDB (clean_shark -> rich_issues)...
Buscando documentos...
Dividindo 1000 documentos...
Total gerado: 2766 chunks.
Iniciando vetorização (Lotes de 32)...


Vetorizando 'issues': 100%|██████████| 87/87 [32:47<00:00, 22.61s/it]


Ingestão na coleção 'issues' concluída com sucesso!

Conectando ao MongoDB (clean_shark -> rich_emails)...
Buscando documentos...
Dividindo 1000 documentos...
Total gerado: 3260 chunks.
Iniciando vetorização (Lotes de 32)...


Vetorizando 'emails': 100%|██████████| 102/102 [1:07:48<00:00, 39.89s/it]


Ingestão na coleção 'emails' concluída com sucesso!

✅ PIPELINE TOTAL CONCLUÍDO!


### 🔍 Busca Semântica

In [7]:
import sys
sys.path.append('./src') # Aponta o notebook para a pasta src

from Config import Config
from VectorStoreManager import VectorStoreManager

print("--- Inicializando o Gerenciador de Banco Vetorial ---")

project = "tez"
db_path = f"./vectorstores/{project}_db"

print(f"Projeto selecionado: {project}")

# 1. Instancia o manager apontando para a pasta local do ChromaDB e nomeando a coleção
vsm = VectorStoreManager(
    persist_directory=db_path, 
    collection_name="commits", 
    model_name="jinaai/jina-embeddings-v2-base-code"
)

# 3. Teste de Busca (Retriever)
print("\n--- Testando a Busca Semântica Local ---")
retriever = vsm.get_retriever(k=3)

# Testando uma query
query_teste = "How are Tez runtime internals implemented in terms of task specification and serialization?"

print(f"🔍 Buscando por: '{query_teste}'\n")
documentos_encontrados = retriever.invoke(query_teste)

# Exibe os resultados limpos
for i, doc in enumerate(documentos_encontrados, 1):
    print(f"📄 DOCUMENTO {i} | Origem: {doc.metadata.get('source')}")
    print(f"{doc.page_content[:300]}...\n")

--- Inicializando o Gerenciador de Banco Vetorial ---
Projeto selecionado: tez


c:\projects\infodev\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\mathe\.cache\huggingface\modules\transformers_modules\jinaai\jina-bert-v2-qk-post-norm\3baf9e3ac750e76e8edd3019170176884695fb94\configuration_bert.py:29: UserWarning: optimum is not installed. To use OnnxConfig and BertOnnxConfig, make sure that `optimum` package is installed
  warnings.warn("optimum is not installed. To use OnnxConfig and BertOnnxConfig, make sure that `optimum` package is installed")



--- Testando a Busca Semântica Local ---
Retriever para 'commits' configurado (k=3).
🔍 Buscando por: 'How are Tez runtime internals implemented in terms of task specification and serialization?'

📄 DOCUMENTO 1 | Origem: Commit_1cc83e457b8a86aab801b526c79652fa99e83186
- * See the License for the specific language governing permissions and
- * limitations under the License.
- */
-
-package org.apache.tez.engine.newapi.rpc.impl;
-
-import java.util.List;
-
-import org.apache.tez.dag.api.ProcessorDescriptor;
-import org.apache.tez.dag.records.TezTaskAttemptID;
-
-/*...

📄 DOCUMENTO 2 | Origem: Commit_2e2312646d25ac0fc40b9cca443f83d3e10b392b
--- File: tez-runtime-internals/src/main/java/org/apache/tez/runtime/api/impl/TaskSpec.java ---
-    // TODO ZZZ Intern this....

📄 DOCUMENTO 3 | Origem: Commit_1616d2c590c23f9f73ea69df0e6d6f95c1960222
- * See the License for the specific language governing permissions and
- * limitations under the License.
- */
-
-package org.apache.tez.engine.newapi.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

print("\n --- Iniciando o Agente Gerador (Llama 3) ---")

# 1. Preparamos o LLM (Usamos uma temperatura baixa, 0.1, para ele não alucinar e focar nos dados)
gerador_llm = ChatGroq(
    model="llama-3.3-70b-versatile", 
    temperature=0.1
)

# 2. Criamos o Prompt de Geração (O "Cérebro" do Bibliotecário)
# Instruímos o modelo a usar APENAS o contexto fornecido. Se não souber, deve dizer que não sabe.
template_geracao = """You are an expert software engineering assistant.
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise and technical.

Question: {question} 

Context:
{context}

Answer:"""

prompt_geracao = ChatPromptTemplate.from_template(template_geracao)

# Extrai apenas o texto puro (page_content) e junta tudo com duas quebras de linha
contexto_formatado = "\n\n".join([doc.page_content for doc in documentos_encontrados])

# 4. Montamos a "Chain" (Corrente) de execução: Prompt -> LLM -> Texto Puro
rag_chain = prompt_geracao | gerador_llm | StrOutputParser()

print(f"🧠 Gerando resposta baseada em {len(documentos_encontrados)} documento(s)...\n")

# 5. Executamos!
resposta_final = rag_chain.invoke({
    "context": contexto_formatado,
    "question": query_teste
})

print("✨ RESPOSTA FINAL DO SISTEMA:")
print("-" * 60)
print(resposta_final)
print("-" * 60)


 --- Iniciando o Agente Gerador (Llama 3) ---
🧠 Gerando resposta baseada em 3 documento(s)...

✨ RESPOSTA FINAL DO SISTEMA:
------------------------------------------------------------
Tez runtime internals implement task specification through the `TaskSpec` interface, which is serializable and sent from the Tez AM to the Tez Container's JVM. The `TaskSpec` interface contains a `ProcessorDescriptor` that defines the processor for a given task. Task specification and serialization are achieved through Java's built-in serialization mechanism, allowing `TaskSpec` objects to be sent across the Umbilical.
------------------------------------------------------------
